# Experiment 9 DL — Temporal Fusion Transformer market forecast

This experiment uses a **global Temporal Fusion Transformer (TFT)** to forecast monthly mean log-rent for `TOWN × FLAT_TYPE` market segments. It deliberately does not fit a neural model per listing or per block: the labelled data cover only 51 calendar months, and the fine-grained property series are too sparse.

Evaluation is strictly chronological. The validation origin is 2024-04: training targets stop at 2024-03 and the model forecasts the next 12 months (2024-04 to 2025-03). The final model uses targets strictly before the first test month (2025-03) and forecasts all 17 test months. This intentionally does not use the labelled 2025-03 rows, since test also contains that month.

TFT forecasts the market component only. It can subsequently be blended with, or used as a feature for, the existing listing-level `compact` model; this notebook does not claim that a segment mean alone replaces the listing model.

## Dependencies

The project environment currently has PyTorch but not PyTorch Forecasting/Lightning. Run the next cell once in the notebook kernel if necessary. The exact compatible package versions can depend on the Python/PyTorch runtime.

In [10]:
# %pip install -U pytorch-forecasting lightning
# Restart the kernel after installation, then continue.

In [11]:
from pathlib import Path
import os
import sys
import random

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping
except ImportError:  # compatibility with older PyTorch Forecasting installs
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import RMSE

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'data/train.csv').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the repository root.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from preprocessing import categorical_clean, load_datasets

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
pl.seed_everything(SEED, workers=True)

# The test horizon is 17 months. Validation is shorter because only 12 fully future
# labelled months are available after the validation origin.
VALIDATION_ORIGIN = pd.Timestamp('2024-04-01')
VALIDATION_HORIZON = 12
MAX_ENCODER_LENGTH = 24
MIN_OBSERVED_HISTORY_MONTHS = 24
BATCH_SIZE = 128
MAX_EPOCHS = 40


Seed set to 42


## Build a regular, leakage-safe segment panel

A TFT requires a regular target history. Missing segment-months are filled from **past values only** (forward fill, then that segment's pre-origin mean); no backward fill is used. Actual observed target values are retained separately and are the only values scored. COE values are treated as known future covariates because the supplied auxiliary data extend through the test horizon.

In [12]:
data_dir, datasets = load_datasets(project_root / 'data')
raw_train = categorical_clean(datasets['train'], columns=['TOWN', 'FLAT_TYPE'], context='TFT inputs', log=False)
raw_test = categorical_clean(datasets['test'], columns=['TOWN', 'FLAT_TYPE'], context='TFT inputs', log=False)
raw_train['period'] = pd.to_datetime(raw_train['RENT_APPROVAL_DATE'] + '-01')
raw_test['period'] = pd.to_datetime(raw_test['RENT_APPROVAL_DATE'] + '-01')

def build_coe_covariates(coe):
    month_map = {name: number for number, name in enumerate(
        ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'], 1
    )}
    result = coe.copy()
    result['period'] = pd.to_datetime(dict(year=result['YEAR'], month=result['MONTH'].str.lower().map(month_map), day=1))
    for column in ['QUOTA_PREMIUM', 'BIDS_RECEIVED']:
        result[column] = pd.to_numeric(
            result[column].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
            errors='coerce',
        )
    return result.groupby('period', as_index=False).agg(
        coe_quota_premium_mean=('QUOTA_PREMIUM', 'mean'),
        coe_bids_mean=('BIDS_RECEIVED', 'mean'),
    )

coe_monthly = build_coe_covariates(datasets['coe'])
test_periods = pd.date_range(raw_test['period'].min(), raw_test['period'].max(), freq='MS')
assert len(test_periods) == 17
assert set(test_periods).issubset(set(coe_monthly['period'])), 'COE covariates must cover every test month.'

def make_panel(rows, start, end, min_observed_months, target_source_end):
    """Return a monthly TOWN x FLAT_TYPE panel with no future-target imputation."""
    segment_columns = ['TOWN', 'FLAT_TYPE']
    # Keep all actual values for later scoring, but use only pre-origin values
    # for eligibility and target-history imputation.
    source = rows[rows['period'].between(start, end)].copy()
    observed = (source.groupby(segment_columns + ['period'], as_index=False)['MONTHLY_RENT']
        .mean().rename(columns={'MONTHLY_RENT': 'rent'}))
    observed['series_id'] = observed['TOWN'].astype(str) + ' | ' + observed['FLAT_TYPE'].astype(str)
    history_observed = observed[observed['period'] <= target_source_end].copy()
    counts = history_observed.groupby(segment_columns)['period'].nunique()
    eligible = counts[counts >= min_observed_months].index
    observed = observed.set_index(segment_columns).loc[eligible].reset_index()
    periods = pd.date_range(start, end, freq='MS')
    series = observed[['series_id', 'TOWN', 'FLAT_TYPE']].drop_duplicates()
    grid = series.merge(pd.DataFrame({'period': periods}), how='cross')
    panel = grid.merge(observed[['series_id', 'period', 'rent']], on=['series_id', 'period'], how='left')
    panel['observed_rent'] = panel['rent']
    panel = panel.sort_values(['series_id', 'period']).reset_index(drop=True)
    # History-only imputation: a missing month may use only earlier observed/imputed months.
    panel['rent'] = panel.groupby('series_id', group_keys=False)['rent'].ffill()
    prior_means = (history_observed
        .groupby('series_id')['rent'].mean())
    panel['rent'] = panel['rent'].fillna(panel['series_id'].map(prior_means))
    panel['time_idx'] = ((panel['period'].dt.year - start.year) * 12 + panel['period'].dt.month - start.month).astype(int)
    panel['month_sin'] = np.sin(2 * np.pi * panel['period'].dt.month / 12)
    panel['month_cos'] = np.cos(2 * np.pi * panel['period'].dt.month / 12)
    panel = panel.merge(coe_monthly, on='period', how='left')
    if panel[['rent', 'coe_quota_premium_mean', 'coe_bids_mean']].isna().any().any():
        raise ValueError('The regular panel contains missing target-history or known-covariate values.')
    panel['log_rent'] = np.log(panel['rent'].clip(lower=1))
    return panel, observed

history_end = VALIDATION_ORIGIN - pd.offsets.MonthBegin(1)
validation_panel, validation_observed = make_panel(
    raw_train, raw_train['period'].min(), VALIDATION_ORIGIN + pd.DateOffset(months=VALIDATION_HORIZON - 1),
    MIN_OBSERVED_HISTORY_MONTHS, history_end,
)
print(f'Validation history ends: {history_end:%Y-%m}; forecast months: {VALIDATION_ORIGIN:%Y-%m} to {(VALIDATION_ORIGIN + pd.DateOffset(months=VALIDATION_HORIZON - 1)):%Y-%m}')
print(f'Eligible market series: {validation_panel.series_id.nunique():,}')


Validation history ends: 2024-03; forecast months: 2024-04 to 2025-03
Eligible market series: 115


## Chronological TFT validation

`training_data` contains only months before the origin. `prediction_data` contains the same encoder history plus forecast rows with a dummy target; TFT does not consume decoder targets when predicting. Thus neither validation rents nor validation aggregates are made available to the model.

In [13]:
def make_dataset(frame, cutoff_time_idx):
    training_frame = frame[frame['time_idx'] <= cutoff_time_idx].copy()
    return TimeSeriesDataSet(
        training_frame,
        time_idx='time_idx', target='log_rent', group_ids=['series_id'],
        min_encoder_length=MAX_ENCODER_LENGTH, max_encoder_length=MAX_ENCODER_LENGTH,
        min_prediction_length=VALIDATION_HORIZON, max_prediction_length=VALIDATION_HORIZON,
        static_categoricals=['series_id', 'TOWN', 'FLAT_TYPE'],
        time_varying_known_reals=['time_idx', 'month_sin', 'month_cos', 'coe_quota_premium_mean', 'coe_bids_mean'],
        time_varying_unknown_reals=['log_rent'],
        target_normalizer=GroupNormalizer(groups=['series_id']),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=False,
    )

origin_idx = int(validation_panel.loc[validation_panel['period'].eq(VALIDATION_ORIGIN), 'time_idx'].iloc[0])
training_data = make_dataset(validation_panel, origin_idx - 1)
prediction_frame = validation_panel.copy()
# Explicitly replace held-out targets, ensuring they cannot become decoder inputs.
prediction_frame.loc[prediction_frame['time_idx'] >= origin_idx, 'log_rent'] = 0.0
validation_data = TimeSeriesDataSet.from_dataset(
    training_data, prediction_frame, predict=True, stop_randomization=True
)

train_loader = training_data.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
valid_loader = validation_data.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)
tft = TemporalFusionTransformer.from_dataset(
    training_data, learning_rate=3e-2, hidden_size=16, attention_head_size=2,
    dropout=0.15, hidden_continuous_size=8, loss=RMSE(), reduce_on_plateau_patience=4,
)
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator='auto', devices=1, gradient_clip_val=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=8, mode='min')],
    enable_model_summary=False, logger=False, enable_checkpointing=False,
)
trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_conne
ctor.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_conne
ctor.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

In [14]:
# `return_index=True` gives the series and decoder start for unambiguous alignment.
# PyTorch Forecasting returns either a Prediction object or a tuple across supported versions.
def predict_with_index(model, loader):
    result = model.predict(loader, mode='prediction', return_index=True)
    if hasattr(result, 'output') and hasattr(result, 'index'):
        return result.output, result.index
    return result

def restore_series_ids(prediction_index, dataset, expected_ids):
    """Handle public and internal PyTorch Forecasting group-id column names."""
    source_column = next(
        (column for column in ['series_id', '__group_id__series_id'] if column in prediction_index.columns),
        None,
    )
    if source_column is None:
        raise KeyError(f'Cannot identify the series column: {prediction_index.columns.tolist()}')
    values = prediction_index[source_column]
    as_text = values.astype(str)
    if as_text.isin(expected_ids).all():
        return as_text
    # Internal group ids are integer-encoded in some releases.
    try:
        encoder = dataset.get_transformer('series_id')
        decoded = pd.Series(encoder.inverse_transform(pd.to_numeric(values).astype(int)), index=values.index).astype(str)
        if decoded.isin(expected_ids).all():
            return decoded
    except (AttributeError, KeyError, TypeError, ValueError):
        pass
    raise KeyError(
        f'Cannot decode forecast series identifiers from {source_column}; '
        f'sample values={values.head().tolist()}, expected sample={list(expected_ids[:3])}'
    )

raw_prediction, prediction_index = predict_with_index(tft, valid_loader)
# Different PyTorch Forecasting releases name this either `time_idx` or
# `time_idx_first_prediction`; both mean the decoder's first time index here.
decoder_time_column = next(
    (column for column in ['time_idx_first_prediction', 'time_idx'] if column in prediction_index.columns),
    None,
)
if decoder_time_column is None:
    raise KeyError(f'Cannot identify the decoder start column: {prediction_index.columns.tolist()}')
expected_series_ids = validation_panel['series_id'].drop_duplicates().astype(str).tolist()
prediction_index = prediction_index.copy()
prediction_index['series_id'] = restore_series_ids(prediction_index, training_data, expected_series_ids)
prediction = np.asarray(raw_prediction.detach().cpu())
forecast_rows = []
for row, (_, index_row) in enumerate(prediction_index.iterrows()):
    series_id = index_row['series_id']
    decoder_start = int(index_row[decoder_time_column])
    for step, log_value in enumerate(prediction[row]):
        forecast_rows.append({'series_id': series_id, 'time_idx': decoder_start + step, 'predicted_rent': float(np.exp(log_value))})
forecast = pd.DataFrame(forecast_rows).merge(
    validation_panel[['series_id', 'time_idx', 'period']].drop_duplicates(), on=['series_id', 'time_idx'], how='left'
)
actual = validation_observed.rename(columns={'rent': 'actual_rent'})[['series_id', 'period', 'actual_rent']]
scored = forecast.merge(actual, on=['series_id', 'period'], how='inner')
metrics = pd.DataFrame([{
    'RMSE': mean_squared_error(scored['actual_rent'], scored['predicted_rent']) ** 0.5,
    'MAE': mean_absolute_error(scored['actual_rent'], scored['predicted_rent']),
    'R2': r2_score(scored['actual_rent'], scored['predicted_rent']),
    'scored_observed_segment_months': len(scored),
    'eligible_segments': validation_panel['series_id'].nunique(),
}])
display(metrics.round(3))
display(scored.sort_values(['period', 'series_id']).head())
metrics.to_csv(experiment_dir / 'exp_9_dl_tft_validation_metrics.csv', index=False)
scored.to_csv(experiment_dir / 'exp_9_dl_tft_validation_forecasts.csv', index=False)


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` 

,RMSE,MAE,R2,scored_observed_segment_months,eligible_segments
0,258.453,182.479,0.84,1340,115


,series_id,time_idx,predicted_rent,period,actual_rent
0,ang mo kio | 2 room,39,1797.532593,2024-04-01,2000.000000
12,ang mo kio | 3 room,39,2566.555176,2024-04-01,2600.892857
24,ang mo kio | 4 room,39,3130.542725,2024-04-01,3379.534884
36,ang mo kio | 5 room,39,3475.829102,2024-04-01,3285.000000
48,bedok | 2 room,39,1783.533813,2024-04-01,2050.000000


## Refit before the test origin and forecast 17 months

Only run this after deciding from the chronological validation result that TFT is worth carrying forward. The output is a segment-month forecast table, not a competition submission. Join it to listings by cleaned `TOWN`, `FLAT_TYPE`, and approval month, then evaluate a blend against the existing `compact` prediction using another rolling origin.

In [15]:
TEST_ORIGIN = raw_test['period'].min()
TEST_HORIZON = len(test_periods)
assert TEST_ORIGIN == pd.Timestamp('2025-03-01') and TEST_HORIZON == 17
final_history_end = TEST_ORIGIN - pd.offsets.MonthBegin(1)
final_panel, _ = make_panel(
    raw_train, raw_train['period'].min(), raw_test['period'].max(),
    MIN_OBSERVED_HISTORY_MONTHS, final_history_end,
)
final_origin_idx = int(final_panel.loc[final_panel['period'].eq(TEST_ORIGIN), 'time_idx'].iloc[0])
final_training_data = make_dataset(final_panel, final_origin_idx - 1)
# Recreate with the actual 17-month decoder length for the test forecast.
final_training_data = TimeSeriesDataSet(
    final_panel[final_panel['time_idx'] < final_origin_idx].copy(),
    time_idx='time_idx', target='log_rent', group_ids=['series_id'],
    min_encoder_length=MAX_ENCODER_LENGTH, max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=TEST_HORIZON, max_prediction_length=TEST_HORIZON,
    static_categoricals=['series_id', 'TOWN', 'FLAT_TYPE'],
    time_varying_known_reals=['time_idx', 'month_sin', 'month_cos', 'coe_quota_premium_mean', 'coe_bids_mean'],
    time_varying_unknown_reals=['log_rent'],
    target_normalizer=GroupNormalizer(groups=['series_id']),
    add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True, allow_missing_timesteps=False,
)
final_prediction_frame = final_panel.copy()
final_prediction_frame.loc[final_prediction_frame['time_idx'] >= final_origin_idx, 'log_rent'] = 0.0
final_prediction_data = TimeSeriesDataSet.from_dataset(
    final_training_data, final_prediction_frame, predict=True, stop_randomization=True
)
final_train_loader = final_training_data.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
final_predict_loader = final_prediction_data.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)
final_tft = TemporalFusionTransformer.from_dataset(
    final_training_data, learning_rate=3e-2, hidden_size=16, attention_head_size=2, dropout=0.15,
    hidden_continuous_size=8, loss=RMSE(), reduce_on_plateau_patience=4,
)
final_trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator='auto', devices=1, gradient_clip_val=0.1,
    enable_model_summary=False, logger=False, enable_checkpointing=False)
final_trainer.fit(final_tft, train_dataloaders=final_train_loader)
final_raw_prediction, final_prediction_index = predict_with_index(final_tft, final_predict_loader)
final_decoder_time_column = next(
    (column for column in ['time_idx_first_prediction', 'time_idx'] if column in final_prediction_index.columns),
    None,
)
if final_decoder_time_column is None:
    raise KeyError(f'Cannot identify the decoder start column: {final_prediction_index.columns.tolist()}')
final_expected_series_ids = final_panel['series_id'].drop_duplicates().astype(str).tolist()
final_prediction_index = final_prediction_index.copy()
final_prediction_index['series_id'] = restore_series_ids(
    final_prediction_index, final_training_data, final_expected_series_ids
)
final_prediction = np.asarray(final_raw_prediction.detach().cpu())
final_rows = []
for row, (_, index_row) in enumerate(final_prediction_index.iterrows()):
    for step, log_value in enumerate(final_prediction[row]):
        final_rows.append({'series_id': index_row['series_id'], 'time_idx': int(index_row[final_decoder_time_column]) + step,
                           'PREDICTED_MARKET_RENT': float(np.exp(log_value))})
final_forecast = pd.DataFrame(final_rows).merge(
    final_panel[['series_id', 'TOWN', 'FLAT_TYPE', 'time_idx', 'period']].drop_duplicates(),
    on=['series_id', 'time_idx'], how='left'
).sort_values(['period', 'series_id'])
output_path = experiment_dir / 'exp_9_dl_tft_market_forecasts.csv'
final_forecast.to_csv(output_path, index=False)
print(f'Saved {len(final_forecast):,} TFT segment-month forecasts to {output_path}')
display(final_forecast.head())


/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/loops/training_epoch_loop.py:
500: ReduceLROnPlateau conditioned on metric val_loss which is not available but strict is set to `False`. Skipping
learning rate update.

`Trainer.fit` stopped: `max_epochs=40` reached.


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/metul/Desktop/KDDM-project/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


Saved 2,057 TFT segment-month forecasts to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_9_dl_tft_market_forecasts.csv


,series_id,time_idx,PREDICTED_MARKET_RENT,TOWN,FLAT_TYPE,period
0,ang mo kio | 2 room,50,1937.729126,ang mo kio,2 room,2025-03-01
17,ang mo kio | 3 room,50,2649.281738,ang mo kio,3 room,2025-03-01
34,ang mo kio | 4 room,50,3230.178467,ang mo kio,4 room,2025-03-01
51,ang mo kio | 5 room,50,3590.369385,ang mo kio,5 room,2025-03-01
68,ang mo kio | executive,50,3900.893555,ang mo kio,executive,2025-03-01
